In [3]:
import math
import os
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import datasets
import einops
import numpy as np
import torch as t
import torch.nn as nn0
import wandb
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import gelu_new, tokenize_and_concatenate
from transformers import GPT2TokenizerFast

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


In [4]:
gpt = HookedTransformer.from_pretrained(
    "gpt2-small",
    fold_ln=False, 
    center_unembed=False,
    center_writing_weights=False  # you'll learn about these arguments later!
)

sorted_vocab = sorted(list(gpt.tokenizer.vocab.items()), key=lambda n: n[1])

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


input = tokens are integer

In [7]:
# gpt.tokenizer.vocab["Ġthe"]
text = "The cat sat on the mat."

tokens = gpt.to_tokens(text)
print(tokens)
print(tokens.shape)
print(gpt.to_str_tokens(tokens))

tensor([[50256,   464,  3797,  3332,   319,   262,  2603,    13]],
       device='mps:0')
torch.Size([1, 8])
['<|endoftext|>', 'The', ' cat', ' sat', ' on', ' the', ' mat', '.']


tokens to logits(output)
- tokens -> h -> output

In [8]:
logits, cache = gpt.run_with_cache(tokens) 
print(logits.shape)
print(cache.keys())

torch.Size([1, 8, 50257])
dict_keys(['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.ln1.hook_scale', 'blocks.1.ln1.hook_normalized', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_attn_scores', 'blocks.1.attn.hook_pattern', 'blocks.1.attn.hook_z', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_mid', 'blocks.1.ln2.hook_scale', 'blocks.1.ln2.hook_normalized', 'blocks.1.mlp.hook_pre', 'blocks.1.mlp.hook_post', 'blocks.1.hook_mlp_out', 'blocks.1.hook_resid_post', 'blocks

logit to prob distribution = unembed

In [9]:
probs = logits.softmax(dim=-1) # 가장 안쪽 
print(probs.shape)

torch.Size([1, 8, 50257])


distribution to token (vector to integer)

In [10]:
next_token = logits[0, -1].argmax(dim=-1)
next_char = gpt.to_string(next_token)
# next_char
print(repr(next_char))

'\n'


add to the end of input + re-run

In [14]:
print(f"Sequence so far: {gpt.to_string(tokens)[0]!r}")

for i in range(10):

    print(f"{tokens.shape[-1] + 1}th char = {next_char!r}")

    tokens = t.cat([tokens, next_token[None, None]], dim=-1) # Define new input sequence, by appending the previously generated token
    logits = gpt(tokens) # Pass our new sequence through the model, to get new output
    next_token = logits[0, -1].argmax(dim=-1) # Get the predicted token at the end of our sequence
    next_char = gpt.to_string(next_token) # Decode and print the result

Sequence so far: '<|endoftext|>The cat sat on the mat.\n\n"I\'m sorry, I\'m sorry," she said.\n\n"I\'m sorry, I\'m sorry," he said.\n\n'
39th char = '"'
40th char = 'I'
41th char = "'m"
42th char = ' sorry'
43th char = ','
44th char = ' I'
45th char = "'m"
46th char = ' sorry'
47th char = ',"'
48th char = ' she'


## Implementation

Embedding
- inputs are tokens(integer)
- to convert integer to vector, we use look-up table. (discrete latent representation)

Positional encoding
- without PE, all tokens are treated equally, as a set. 
- trick to break `permutation invariance` 
- i.e, to give a sensitivity of "order" of input elements. 
- we dunno whether this information survives til finla representation


### Transformer block * `n_layers`
- Attention head
- Layer Norm
- Residual Stream
- MLP

Attention 
- Attention is aggregation function s.t. way of summing up informations from neighbourhood. In graph manner, we call it message passing. [Transformers are GNN](https://thegradient.pub/transformers-are-graph-neural-networks/).
- By mixing oneself with neighbours (i.e, contextual info), one can get richer infomation. 
- View : sentence is a fully-connected graph. set of node(token) with edges (underlying relationship).


Attention Head
- independent. but informations can be shared/mixed indirectly through residual stream. 
- info by each head is aggregated to residual stream in additive manner.

Layer Norm
- for training stabilization. = training takes too long. weight are not updated properly!
- prevent **gradient vanish or explode.**

1. partial derivatives die. activation functions are usally flat.: before activation function.
    - $\frac{\partial L}{\partial x} = \prod_{l=1}^{L} J_l  = \prod_{l=1}^{L} \frac{\partial h_l}{\partial h_{l-1}}$ 
    - Matrix 를 연쇄적으로 곱할 떄, 어디로 수렴할지는 Jacobian Norm 으로 판단 가능. 
    - $\prod \|J_l\| \to 0 \text{ or } \infty$ 이때 연쇄 곱의 대상은 각 레이어!

2. scale of activation matters > cus it directly scales gradient > which makes learning step useless.
    - $h_{l+1} = h_l + f(h_l)$, $\|h_l\| \sim O(l)$

1. parameter distribution shift. - 학습하다보면 이전 레이어의 activation distribution이 변한다는데 ,,, 먼 솔


Residual Stream
- high-dim vector space.
- each module (attn, mlp) read info from this stream and write back. 
- informations interact!
- It's like a memory.


MLP
- $W_{in} \to \text{GELU} \to W_{out}$
- responsible for mixing? look up information? > Yet discovered!

### Unembed = final output of transformer are `logits`.

# Code Implementation

activations are different from parameters (weight)
- **weight/param** is fixed, deterministic after training. 
- **activations** are function of input. it's like flash state dependent to input. 
    - This temporal state only alive during forward pass. thats why we need hooking function to grab this
    - attentions are also activations. 


- [diagram of params](https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-full-updated.png).

In [15]:
print(f"Size of activations during the forward pass  (input : {text})")
print()
for activation_name, activation in cache.items():
    # Only print for first layer
    if ".0." in activation_name or "blocks" not in activation_name:
        print(f"{activation_name:30} {tuple(activation.shape)}")


print("="*50)

print("Size of parameters. can only change/updated through training. never change after training")
print()

for name, param in gpt.named_parameters():
    # Only print for first layer
    if ".0." in name or "blocks" not in name:
        print(f"{name:18} {tuple(param.shape)}")

Size of activations during the forward pass  (input : The cat sat on the mat.)

hook_embed                     (1, 8, 768)
hook_pos_embed                 (1, 8, 768)
blocks.0.hook_resid_pre        (1, 8, 768)
blocks.0.ln1.hook_scale        (1, 8, 1)
blocks.0.ln1.hook_normalized   (1, 8, 768)
blocks.0.attn.hook_q           (1, 8, 12, 64)
blocks.0.attn.hook_k           (1, 8, 12, 64)
blocks.0.attn.hook_v           (1, 8, 12, 64)
blocks.0.attn.hook_attn_scores (1, 12, 8, 8)
blocks.0.attn.hook_pattern     (1, 12, 8, 8)
blocks.0.attn.hook_z           (1, 8, 12, 64)
blocks.0.hook_attn_out         (1, 8, 768)
blocks.0.hook_resid_mid        (1, 8, 768)
blocks.0.ln2.hook_scale        (1, 8, 1)
blocks.0.ln2.hook_normalized   (1, 8, 768)
blocks.0.mlp.hook_pre          (1, 8, 3072)
blocks.0.mlp.hook_post         (1, 8, 3072)
blocks.0.hook_mlp_out          (1, 8, 768)
blocks.0.hook_resid_post       (1, 8, 768)
ln_final.hook_scale            (1, 8, 1)
ln_final.hook_normalized       (1, 8, 768)
Size 

In [16]:
# print(gpt.cfg)

In [17]:
@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12

cfg = Config()
print(cfg)

Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)


Test with random input

In [18]:
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    random_input = t.randn(shape).to(device)
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape, "\n")


def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    random_input = t.randint(100, 1000, shape).to(device)
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape, "\n")

In [ ]:
def load_gpt_test(cls, gpt_layer, input):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    layer.load_state_dict(gpt_layer.state_dict(), strict=False)
    print("Input shape:", input.shape)
    orig_input = input.clone()
    output = layer(orig_input)


    ## 로직 구현 오류 (Implementation Bugs). 실제 레퍼런스 모델이 뱉는 값이랑 다르게 나오지 않도록 
    
    assert t.allclose(input, orig_input), "Input has been modified, make sure operations are not done in place"
    if isinstance(output, tuple):
        output = output[0]
    # print("Output shape:", output.shape)
    try:
        reference_output = gpt_layer(input)
    except:
        reference_output = gpt_layer(input, input, input)
    print("Reference output shape:", reference_output.shape, "\n")
    comparison = t.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum() / comparison.numel():.2%} of the values are correct\n")
    assert 1 - (comparison.sum() / comparison.numel()) < 1e-5, "More than 0.01% of the values are incorrect"

## Implementation 

* The `layer_norm_eps` argument in your config object corresponds to the $\epsilon$ term in the PyTorch documentation (it is included to avoid division-by-zero errors).
* We've given you a `debug` argument in your config. If `debug=True`, then you can print output like the shape of objects in your `forward` function to help you debug (this is a very useful trick to improve your coding speed).

Fill in the function, where it says `raise NotImplementedError()` (this will be the basic pattern for most other exercises in this section).

In [44]:
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(t.ones(cfg.d_model))
        self.b = nn.Parameter(t.zeros(cfg.d_model))

    def forward(self, residual: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        res_mean = residual.mean(dim=-1, keepdim=True)
        res_std = (residual.var(dim=-1, keepdim=True, unbiased=False) + self.cfg.layer_norm_eps).sqrt()

        res = (residual - res_mean) / res_std

        res = self.w * res + self.b
        return res

In [ ]:
# W_E = gpt.embed.state_dict()['W_E']

In [ ]:
# W_E.shape

# W_E[[2,4]]

tensor([[-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        [-0.0506, -0.1111,  0.1058,  ..., -0.1149,  0.0664,  0.0574]],
       device='mps:0')

In [87]:
class Embed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(t.empty((cfg.d_vocab, cfg.d_model))) # 빈텐서 할당
        nn.init.normal_(self.W_E, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:

        return self.W_E[tokens]


rand_int_test(Embed, [2, 4]) # (추출하고 싶은 행의 번호들 리스트로)
load_gpt_test(Embed, gpt.embed, tokens)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 768]) 

Reference output shape: torch.Size([1, 8, 768]) 

100.00% of the values are correct



In [ ]:
# W = gpt.pos_embed.state_dict()['W_pos']

# b , seq_len = tokens.shape

# b, seqlen

# W[:seq_len].shape

torch.Size([8, 768])

In [104]:
class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        batch , seq_len = tokens.shape
        return einops.repeat(self.W_pos[:seq_len], "seq d_model -> batch seq d_model", batch=batch) ## 배치/문장이 달라져도 더해줘야하는 PE는 동일!


rand_int_test(PosEmbed, [2, 4])
load_gpt_test(PosEmbed, gpt.pos_embed, tokens)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 768]) 

Reference output shape: torch.Size([1, 8, 768]) 

100.00% of the values are correct



In [ ]:
# class Attention(nn.Module):
#     IGNORE: Float[Tensor, ""]

#     def __init__(self, cfg: Config):
#         super().__init__()
#         self.cfg = cfg
#         self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

#     def apply_causal_mask(
#         self,
#         attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"],
#     ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        

#         attn_scores = attn_scores.masked_fill(
#         """
#         Applies a causal mask to attention scores, and returns masked scores.
#         """
#         raise NotImplementedError()


# # tests.test_causal_mask(Attention.apply_causal_mask)

In [106]:
cfg.n_ctx

1024

In [ ]:
mask = t.triu(t.ones(seq_len,n_ctx), diagonal=1)

mask

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.]])

In [109]:
scores = t.randn(3, 3)
mask = t.triu(t.ones(3, 3), diagonal=1).bool()
mask
# masked_scores = scores.masked_fill(mask, -1e9) # 미래는 -9억점으로 만듦

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [ ]:
rand_int_test(LayerNorm, (2, 4, 768))
load_gpt_test(LayerNorm, gpt.blocks[0].ln1, t.randn(2, 4, 768).to(device))

In [47]:
LayerNorm(Config(debug=True)).cfg

Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)

In [ ]:
cfg = Config(debug=True)

layer = LayerNorm(cfg).to(device)






In [61]:
gpt.ln_final.state_dict()['w'].shape

torch.Size([768])

In [ ]:
layer.load_state_dict(gpt_layer.state_dict(), strict=False)

In [ ]:
def load_gpt_test(cls, gpt_layer, input):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    layer.load_state_dict(gpt_layer.state_dict(), strict=False)

    # print("Input shape:", input.shape)

    orig_input = input.clone()
    output = layer(orig_input)





    assert t.allclose(input, orig_input), "Input has been modified, make sure operations are not done in place"
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape)
    try:
        reference_output = gpt_layer(input)
    except:
        reference_output = gpt_layer(input, input, input)
    print("Reference output shape:", reference_output.shape, "\n")
    comparison = t.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum() / comparison.numel():.2%} of the values are correct\n")
    assert 1 - (comparison.sum() / comparison.numel()) < 1e-5, "More than 0.01% of the values are incorrect"

In [43]:
# def f(res: Float[Tensor, "batch seq_len d_model"]) -> Float[Tensor, "batch seq_len d_model"]:
#     res.
#     print(res.shape)
#     return res


x= t.randn(1,2,3) # t.randn(2,3,4 )
# res.abs()


# x의 모양: [2, 3, 4]

# 1. keepdim=False 일 때 (에러 발생 가능성 높음)
mean = x.mean(dim=-1, keepdim=False) # [2, 3]
# x - mean  <-- [2, 3, 4]와 [2, 3]은 모양이 안 맞아서 연산 에러!
print(mean.shape)

# 2. keepdim=True 일 때 (매우 편리함)
mean = x.mean(dim=-1, keepdim=True)  # [2, 3, 1]
# x - mean  <-- [2, 3, 4]와 [2, 3, 1]은 브로드캐스팅(Broadcasting) 덕분에 자동으로 연산됨!
print(mean.shape)


torch.Size([1, 2])
torch.Size([1, 2, 1])


LayerNorm(
  (hook_scale): HookPoint()
  (hook_normalized): HookPoint()
)

In [ ]:

# def rand_int_test(cls, shape):
shape = [2, 4, 768]
cfg = Config(debug=True)
layer = cls(cfg).to(device)
random_input = t.randint(100, 1000, shape).to(device)
print("Input shape:", random_input.shape)
output = layer(random_input)
if isinstance(output, tuple):
    output = output[0]
print("Output shape:", output.shape, "\n")

rand_float_test(LayerNorm, [2, 4, 768])
load_gpt_test(LayerNorm, gpt.ln_final, cache["resid_post", 11])

# tests.test_layer_norm_epsilon(LayerNorm, cache["resid_post", 11])